In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from art.estimators.classification import XGBoostClassifier as ARTXGBoost
from art.attacks.evasion import HopSkipJump

In [2]:
df = pd.read_csv("clean_sample_150000.csv")
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])
X = df.drop(columns=["label"], errors="ignore").select_dtypes(include=[np.number])
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0, inplace=True)
y = df["label"]
print("Classes:", dict(zip(le.classes_, range(len(le.classes_)))))
print("Shape:", X.shape)

Classes: {'BENIGN': 0, 'DoS_ATTACK': 1, 'NON_DoS_ATTACK': 2}
Shape: (150000, 88)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.astype("float32").reset_index(drop=True)
X_test  = X_test.astype("float32").reset_index(drop=True)
y_train = y_train.reset_index(drop=True).to_numpy()
y_test  = y_test.reset_index(drop=True).to_numpy()

model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.03,
    eval_metric="mlogloss", random_state=42, tree_method="hist"
)
model.fit(X_train, y_train, verbose=False)
y_pred = model.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))
print("Baseline Macro F1:", f1_score(y_test, y_pred, average="macro", zero_division=0))

Baseline Accuracy: 0.9070666666666667
Baseline Macro F1: 0.9069017276411162


In [4]:
n_classes = len(le.classes_)
art_classifier = ARTXGBoost(
    model=model,
    nb_features=X_train.shape[1],
    nb_classes=n_classes,
    clip_values=(X_train.values.min(), X_train.values.max()),
)

In [5]:
# Use a subset of the training set to keep runtime manageable.
# Increase N_SAMPLES for a larger adversarial CSV.
N_SAMPLES_PER_CLASS = 150000

attack_mask = y_train != 0          # exclude BENIGN
correct_mask = model.predict(X_train) == y_train
candidate_mask = attack_mask & correct_mask

rng = np.random.default_rng(42)
selected_idx = []
for cls in np.unique(y_train[attack_mask]):
    cls_idx = np.where(candidate_mask & (y_train == cls))[0]
    chosen = rng.choice(cls_idx, size=min(N_SAMPLES_PER_CLASS, len(cls_idx)), replace=False)
    selected_idx.extend(chosen.tolist())

X_attack = X_train.iloc[selected_idx].values.astype("float32")
y_attack  = y_train[selected_idx]
print(f"Attack pool: {len(X_attack)} samples")
print("Per class:", dict(zip(*np.unique(y_attack, return_counts=True))))

Attack pool: 69128 samples
Per class: {1: 32958, 2: 36170}


## Run HopSkipJump attack

Parameters:
- `targeted=False` — untargeted: just fool the classifier away from the true label
- `norm=2` — L2 perturbation budget (use `np.inf` for L∞)
- `max_iter` / `max_eval` — trade off quality vs. runtime; increase for stronger attacks

In [6]:
attack = HopSkipJump(
    classifier=art_classifier,
    targeted=False,
    norm=2,
    max_iter=20,
    max_eval=1000,
    init_eval=100,
    init_size=100,
    verbose=True,
)

X_adv = attack.generate(x=X_attack)

HopSkipJump:   0%|          | 0/69128 [00:00<?, ?it/s]

## Evaluate attack success rate

In [7]:
y_pred_clean = model.predict(X_attack)
y_pred_adv   = model.predict(X_adv)

fooled = y_pred_adv != y_attack
asr = fooled.mean()

# L2 perturbation magnitude per sample
l2_norms = np.linalg.norm(X_adv - X_attack, axis=1)

print(f"Attack Success Rate : {asr*100:.1f}%  ({fooled.sum()}/{len(fooled)} samples fooled)")
print(f"Mean L2 perturbation: {l2_norms.mean():.4f}")
print(f"Median L2           : {np.median(l2_norms):.4f}")
print()

# Only include labels that actually appear across true + predicted sets
present_labels = np.unique(np.concatenate([y_attack, y_pred_clean, y_pred_adv]))
present_names  = le.classes_[present_labels]

print("Clean predictions:")
print(classification_report(y_attack, y_pred_clean, labels=present_labels, target_names=present_names, zero_division=0))
print("Adversarial predictions:")
print(classification_report(y_attack, y_pred_adv, labels=present_labels, target_names=present_names, zero_division=0))

Attack Success Rate : 52.3%  (36170/69128 samples fooled)
Mean L2 perturbation: 52.8256
Median L2           : 0.0035

Clean predictions:
                precision    recall  f1-score   support

        BENIGN       0.00      0.00      0.00         0
    DoS_ATTACK       1.00      1.00      1.00     32958
NON_DoS_ATTACK       1.00      1.00      1.00     36170

      accuracy                           1.00     69128
     macro avg       0.67      0.67      0.67     69128
  weighted avg       1.00      1.00      1.00     69128

Adversarial predictions:
                precision    recall  f1-score   support

        BENIGN       0.00      0.00      0.00         0
    DoS_ATTACK       0.48      1.00      0.65     32958
NON_DoS_ATTACK       0.00      0.00      0.00     36170

      accuracy                           0.48     69128
     macro avg       0.16      0.33      0.22     69128
  weighted avg       0.23      0.48      0.31     69128



## Export successful adversarial samples

Output CSV matches the schema of `adversarial_epsilon_*.csv` from the DNN pipeline
so `poison_training_set` in RQ1.ipynb can consume it directly.

In [8]:
# Keep only successfully fooled samples, mislabeled as BENIGN (label 0)
success_mask = fooled & (y_pred_adv == 0)   # fooled AND predicted as BENIGN

adv_df = pd.DataFrame(X_adv[success_mask], columns=X_train.columns)
adv_df["true_label"] = y_attack[success_mask]
adv_df["pred_clean"] = y_pred_clean[success_mask]
adv_df["pred_adv"]   = y_pred_adv[success_mask]

# Balance classes
min_count = adv_df["true_label"].value_counts().min()
adv_df = (
    adv_df.groupby("true_label", group_keys=False)
          .apply(lambda g: g.sample(min(min_count, len(g)), random_state=42))
          .reset_index(drop=True)
)

adv_df.to_csv("adversarial_hsj_xgb.csv", index=False)
print(f"Exported {len(adv_df)} samples ({min_count} per class) → adversarial_hsj_xgb.csv")
print("Class breakdown:", adv_df["true_label"].value_counts().to_dict())

Exported 1115 samples (1115 per class) → adversarial_hsj_xgb.csv
Class breakdown: {2: 1115}


C:\Users\Aristophanies\AppData\Local\Temp\ipykernel_33428\3566700052.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(min_count, len(g)), random_state=42))


In [9]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

dnn_csv = "adversarial_epsilon_5.csv"
if os.path.exists(dnn_csv):
    dnn_df = pd.read_csv(dnn_csv)
    feature_cols = X_train.columns.tolist()
    dnn_feat_cols = [c for c in feature_cols if c in dnn_df.columns]

    # Sample matching count for fair comparison
    n = min(len(adv_df), len(dnn_df))
    dnn_sample = dnn_df[dnn_feat_cols].iloc[:n].values.astype("float32")
    hsj_sample = adv_df[dnn_feat_cols].values.astype("float32")

    # Use original clean samples as reference for DNN norms
    orig_clean = X_train[dnn_feat_cols].iloc[:n].values.astype("float32")
    dnn_norms = np.linalg.norm(dnn_sample - orig_clean, axis=1)
    hsj_norms = np.linalg.norm(hsj_sample - X_attack[success_mask][:, :len(dnn_feat_cols)], axis=1)

    print(f"DNN (ε=5) median L2 : {np.median(dnn_norms):.2f}")
    print(f"HSJ XGB   median L2 : {np.median(hsj_norms):.2f}")

    plt.figure(figsize=(8, 4))
    plt.hist(dnn_norms, bins=50, alpha=0.6, label=f"DNN PGD ε=5 (median={np.median(dnn_norms):.1f})")
    plt.hist(hsj_norms, bins=50, alpha=0.6, label=f"HopSkipJump XGB (median={np.median(hsj_norms):.1f})")
    plt.xlabel("L2 perturbation magnitude")
    plt.ylabel("Count")
    plt.title("Perturbation Distribution: DNN-crafted vs XGBoost-targeted")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("perturbation_comparison.png", dpi=100)
    plt.show()
    print("Plot saved to perturbation_comparison.png")
else:
    print(f"{dnn_csv} not found — skipping comparison plot.")

ModuleNotFoundError: No module named 'matplotlib.backends.registry'